# P20 a frontier, different-family guide (Claude Opus 5.5, Fable 5.1)
The guide is a Claude model via the Anthropic API; the agent stays Gemma 4 31B. Original P13 prompt, R=3, three stores (clean, P13 wrong lessons, agreeing wrong lessons). Tests whether the instruction-agreement behaviour generalises beyond the two open ~30B judges (Gemma, Qwen). Predictions: `quick_runs/predictions/P20.md`.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks_p12.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
from metrics import decision_agreement, first_divergence, answer_consistency, consistent_failures, score_spread, task_score
from reveries.decisions import DECISION_TYPES
from score import family_of, SF_CHECKS
DTYPES = [t for t in DECISION_TYPES if t != "final_selection"] + ["final_selection"]
def load_records(probe):
    recs = [json.loads(l) for l in open(RUNS / probe / "records.jsonl")]
    for r in recs:
        r["answer"] = tuple(r["answer"]) if isinstance(r["answer"], list) else r["answer"]
        r["risk"] = TASKS[r["task_id"]]["contamination_risk"]
        r["source"] = "SF" if r["task_id"].startswith("sf_") else "BLADE"
    return recs
def by_task(recs):
    d = defaultdict(list)
    for r in recs: d[r["task_id"]].append(r)
    return dict(sorted(d.items()))
def task_table(groups):
    rows = []
    for t, rs in groups.items():
        row = {"task": t, "risk": TASKS[t]["contamination_risk"], "n": len(rs),
               "answer_consistency": answer_consistency(rs), "score_mean": task_score(rs), "score_sd": score_spread(rs),
               "first_divergence": first_divergence(rs, DTYPES),
               "consistent_failures": ",".join(consistent_failures(rs))}
        for dt in DTYPES:
            row[dt] = decision_agreement(rs, dt)
        rows.append(row)
    return pd.DataFrame(rows)

In [3]:
import re
TASKS4 = ["sf_tf01_goal_imbalance", "sf_tf02_player_rep_norm", "sf_tf03_correlation_causation", "sf_tf05_credit_card_leakage"]
SF6 = TASKS4 + ["sf_tf04_balance_torque_leakfree", "sf_tf06_metric_select_leakfree"]
TARGET = {"sf_tf01_goal_imbalance": ["checks_class_distribution", "uses_class_weight", "uses_resampling", "uses_stratified_split"],
          "sf_tf02_player_rep_norm": ["handles_unit_suffix_in_height"], "sf_tf03_correlation_causation": ["column_x_included", "applies_causal_reasoning"],
          "sf_tf05_credit_card_leakage": ["used_leakage_features"]}
WRONG_COL = {"sf_tf01_goal_imbalance": "assists", "sf_tf02_player_rep_norm": "height_in_cm",
             "sf_tf03_correlation_causation": "assists", "sf_tf05_credit_card_leakage": "reports"}
FIX = {"sf_tf05_credit_card_leakage": {164, 166, 167, 168, 169}, "sf_tf03_correlation_causation": {127, 129, 131, 132, 134, 135, 137}}
CONTRA3 = {130, 136}; WRONG_IDS = set(range(708, 720))
INSTR = r"task (says|asks|states|requires|instructs)|instruction|use all|all available|contradict"
T14 = [json.loads(l) for l in open(REPO / "quick_runs/results/P14/guide_trace.jsonl")]
T13 = [json.loads(l) for l in open(REPO / "quick_runs/results/P13/guide_trace.jsonl")]
print(Counter(t["arm"] for t in T14))

Counter({'G3o': 30, 'G3n': 30, 'G3o-p': 20, 'G3n-p': 20})


In [4]:
# Stage 1: fix lessons and wrong lessons, per guide arm (runs out of 5 or 20). P13 G3 / G3-p for reference.
def stage1(trs, arm):
    row = {"arm": arm}
    for t, ids in FIX.items():
        rs = [x for x in trs if x["arm"] == arm.split(" ")[0] and x["task"] == t]
        if not rs: continue
        row[t[:7] + " fix seen"] = sum(bool(set(x["seen"]) & ids) for x in rs)
        row[t[:7] + " fix shown"] = sum(bool(set(x["shown"]) & ids) for x in rs)
        row[t[:7] + " n"] = len(rs)
    if t := [x for x in trs if x["task"] == "sf_tf03_correlation_causation" and x["arm"] == arm.split(" ")[0]]:
        row["tf03 contra shown"] = sum(bool(set(x["shown"]) & CONTRA3) for x in t)
    return row
def wrong(trs, arm):
    rs = [x for x in trs if x["arm"] == arm]
    return {"arm": arm, "runs": len(rs), "wrong seen": sum(bool(set(x["seen"]) & WRONG_IDS) for x in rs),
            "wrong accepted": sum(bool(set(x["accepted"]) & WRONG_IDS) for x in rs), "wrong shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs),
            **{f"{t[:7]} shown": sum(bool(set(x["shown"]) & WRONG_IDS) for x in rs if x["task"] == t) for t in TASKS4}}
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(T14, "G3o"), stage1(T14, "G3n")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(T14, "G3o-p"), wrong(T14, "G3n-p")]).set_index("arm")
print(s1.to_string()); wr

     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3o                 5                  0          5                 5                  1          5                  4
G3n                 5                  2          5                 5                  0          5                  5


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3o-p,20,20,6,6,5,0,0,1
G3n-p,20,20,10,9,5,0,0,4


In [5]:
TC = [json.loads(l) for l in open(REPO / "quick_runs/results/P20/trace_opus.jsonl")]
TF = [json.loads(l) for l in open(REPO / "quick_runs/results/P20/trace_fable.jsonl")]
T16 = [json.loads(l) for l in open(REPO / "quick_runs/results/P16/guide_trace.jsonl")]
for t in TC + TF + T16: t.setdefault("seen", t["shown"]); t.setdefault("accepted", t["shown"])
print(Counter(t["arm"] for t in TC), Counter(t["arm"] for t in TF))
# Stage 1: fix lessons and wrong lessons shown, Claude vs Gemma (G3/G3-p/G3-pc) and Qwen (Q3)
s1 = pd.DataFrame([stage1(T13, "G3"), stage1(TC, "G3c"), stage1(TF, "G3f")]).set_index("arm")
wr = pd.DataFrame([wrong(T13, "G3-p"), wrong(TC, "G3c-p"), wrong(TF, "G3f-p"),
                   wrong(T16, "G3-pc"), wrong(TC, "G3c-pc"), wrong(TF, "G3f-pc")]).set_index("arm")
print(s1.to_string()); wr

Counter({'G3c': 30, 'G3c-p': 20, 'G3c-pc': 20}) Counter({'G3f': 30, 'G3f-p': 20, 'G3f-pc': 20})
     sf_tf05 fix seen  sf_tf05 fix shown  sf_tf05 n  sf_tf03 fix seen  sf_tf03 fix shown  sf_tf03 n  tf03 contra shown
arm                                                                                                                   
G3                  5                  0          5                 5                  1          5                  4
G3c                 5                  0          5                 5                  1          5                  4
G3f                 5                  0          5                 5                  1          5                  4


,runs,wrong seen,wrong accepted,wrong shown,sf_tf01 shown,sf_tf02 shown,sf_tf03 shown,sf_tf05 shown
arm,,,,,,,,
G3-p,20,20,5,5,4,0,0,1
G3c-p,20,20,0,0,0,0,0,0
G3f-p,20,20,0,0,0,0,0,0
G3-pc,20,20,20,20,5,5,5,5
G3c-pc,20,20,1,1,1,0,0,0
G3f-pc,20,20,9,9,3,0,1,5


In [6]:
# Reasons: share of fix-lesson rejections citing the instructions (pre-set regex)
rows = []
for t in TC + TF:
    for r in t.get("rounds", []):
        for j in r.get("judgements") or []:
            try: i = int(j.get("id"))
            except (TypeError, ValueError): continue
            kind = "fix" if i in FIX.get(t["task"], set()) else "wrong" if i in WRONG_IDS else None
            if kind is None or i not in r["candidates"]: continue
            rows.append({"model": "Opus" if t in TC else "Fable", "arm": t["arm"], "kind": kind,
                         "useful": j.get("useful") is True, "instr": bool(re.search(INSTR, str(j.get("reason","")), re.I)),
                         "reason": str(j.get("reason", ""))})
jr = pd.DataFrame(rows)
print(jr[(~jr.useful) & (jr.kind == "fix")].groupby("model").instr.agg(["sum", "count", "mean"]).round(2).to_string())
print("\naccepted/rejected agreeing wrong lessons, examples:")
for _, r in jr[jr.kind == "wrong"].groupby(["model", "useful"]).head(1).iterrows(): print(f"{r.model} useful={r.useful}: {r.reason[:170]}")

       sum  count  mean
model                  
Fable  101    141  0.72
Opus    64    103  0.62

accepted/rejected agreeing wrong lessons, examples:
Opus useful=False: This contradicts the task, which says to use the numerical feature columns, and assists is one of them. Nothing shows that assists leaks goals_label. Dropping it would mo
Opus useful=True: The task fixes an 80/20 split with random_state=42. This note keeps that split exactly as written and does not add stratification, which would change the result. Its clai
Fable useful=False: Conflicts with the task instruction to use all numerical feature columns; assists is a legitimate correlated feature, not label leakage, and dropping it would deviate fro
Fable useful=True: Task fixes the split (80/20, random_state=42); keeping it exactly as requested without stratify is the correct action and avoids diverging from the expected protocol.


In [7]:
# Stage 2: agent runs. Targeted checks on the 4 failing tasks (passes out of 5), P14 G3n next to P12/P13 arms
def load(arm):
    recs = [json.loads(l) for l in open(RUNS / arm / "records.jsonl")]
    recs = [r for r in recs if r["task_id"] in SF6]
    for r in recs:
        ev = [json.loads(l) for l in open(RUNS / arm / r["task_id"] / r["run"] / "events.jsonl")]
        calls = [e for e in ev if e["event"] == "llm_call" and e.get("usage")]
        r["sum_tokens"] = sum(c["usage"]["prompt_tokens"] + c["usage"]["completion_tokens"] for c in calls)
    return recs
ARMS = ["P12_A0", "P13_E", "P13_G3", "P20_G3c", "P20_G3f", "P13_G3p", "P20_G3cp", "P20_G3fp", "P16_G3pc", "P20_G3cpc", "P20_G3fpc"]
NAME = dict(zip(ARMS, ["A0 none", "E emb", "G3 Gemma", "G3c Opus", "G3f Fable", "G3-p", "G3c-p", "G3f-p", "G3-pc", "G3c-pc", "G3f-pc"]))
recs = {a: load(a) for a in ARMS}
print({a: (len(v), dict(Counter(r["stop"] for r in v))) for a, v in recs.items()})
rows = []
for t, checks in TARGET.items():
    for c in checks:
        rows.append({"task": t[:7], "check": c, **{NAME[a]: sum(r["checks"].get(c) is True for r in recs[a] if r["task_id"] == t) for a in ARMS}})
targeted = pd.DataFrame(rows)
pooled = targeted[[NAME[a] for a in ARMS]].sum() / 40
print("pooled:", pooled.round(3).to_dict()); targeted

{'P12_A0': (30, {'final_answer': 30}), 'P13_E': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_G3': (30, {'final_answer': 29, 'max_steps': 1}), 'P20_G3c': (30, {'final_answer': 28, 'max_steps': 2}), 'P20_G3f': (30, {'final_answer': 29, 'max_steps': 1}), 'P13_G3p': (20, {'final_answer': 20}), 'P20_G3cp': (20, {'final_answer': 20}), 'P20_G3fp': (20, {'final_answer': 20}), 'P16_G3pc': (20, {'final_answer': 20}), 'P20_G3cpc': (20, {'final_answer': 20}), 'P20_G3fpc': (20, {'final_answer': 20})}
pooled: {'A0 none': 0.075, 'E emb': 0.475, 'G3 Gemma': 0.5, 'G3c Opus': 0.125, 'G3f Fable': 0.3, 'G3-p': 0.45, 'G3c-p': 0.225, 'G3f-p': 0.125, 'G3-pc': 0.0, 'G3c-pc': 0.525, 'G3f-pc': 0.35}


,task,check,A0 none,E emb,G3 Gemma,G3c Opus,G3f Fable,G3-p,G3c-p,G3f-p,G3-pc,G3c-pc,G3f-pc
0,sf_tf01,checks_class_distribution,0,5,4,0,0,3,0,0,0,1,0
1,sf_tf01,uses_class_weight,0,5,5,0,3,5,0,0,0,2,1
2,sf_tf01,uses_resampling,0,0,0,0,0,0,0,0,0,0,0
3,sf_tf01,uses_stratified_split,3,5,5,1,3,5,2,0,0,4,1
4,sf_tf02,handles_unit_suffix_in_height,0,4,4,3,4,5,5,5,0,5,5
5,sf_tf03,column_x_included,0,0,1,1,1,0,1,0,0,5,4
6,sf_tf03,applies_causal_reasoning,0,0,1,0,1,0,1,0,0,4,3
7,sf_tf05,used_leakage_features,0,0,0,0,0,0,0,0,0,0,0


In [8]:
# Harm, cost (Claude USD from the ledgers), and exact tests
from scipy.stats import fisher_exact
sc = pd.DataFrame([{"arm": NAME[a], "task": r["task_id"], "score": r["score"]} for a in ARMS for r in recs[a]])
scores = sc.pivot_table(index="task", columns="arm", values="score", aggfunc="mean").round(4)
def usd(f):
    try: return round(sum(json.loads(l)["usd"] for l in open(REPO / f)), 2)
    except FileNotFoundError: return None
cost = {"Opus 5.5 guide": usd("quick_runs/results/P20/ledger_opus.jsonl"), "Fable 5.1 guide": usd("quick_runs/results/P20/ledger_fable.jsonl")}
passed = (pooled * 40).round().astype(int)
fp = lambda a, b: round(fisher_exact([[a, 40 - a], [b, 40 - b]])[1], 4)
fisher = {"pooled G3 vs G3c": fp(passed["G3 Gemma"], passed["G3c Opus"]), "pooled G3 vs G3f": fp(passed["G3 Gemma"], passed["G3f Fable"]),
          "agreeing shown G3-pc vs G3c-pc": round(fisher_exact([[int(wr.loc["G3-pc","wrong shown"]),20-int(wr.loc["G3-pc","wrong shown"])],[int(wr.loc["G3c-pc","wrong shown"]),20-int(wr.loc["G3c-pc","wrong shown"])]])[1],4),
          "agreeing shown G3-pc vs G3f-pc": round(fisher_exact([[int(wr.loc["G3-pc","wrong shown"]),20-int(wr.loc["G3-pc","wrong shown"])],[int(wr.loc["G3f-pc","wrong shown"]),20-int(wr.loc["G3f-pc","wrong shown"])]])[1],4)}
print("Claude guide cost (USD):", cost); print(fisher); scores

Claude guide cost (USD): {'Opus 5.5 guide': 4.76, 'Fable 5.1 guide': 16.77}
{'pooled G3 vs G3c': np.float64(0.0006), 'pooled G3 vs G3f': np.float64(0.1095), 'agreeing shown G3-pc vs G3c-pc': np.float64(0.0), 'agreeing shown G3-pc vs G3f-pc': np.float64(0.0001)}


arm,A0 none,E emb,G3 Gemma,G3-p,G3-pc,G3c Opus,G3c-p,G3c-pc,G3f Fable,G3f-p,G3f-pc
task,,,,,,,,,,,
sf_tf01_goal_imbalance,7.603000e-01,7.713000e-01,7.715000e-01,7.449000e-01,7.865000e-01,7.866000e-01,7.890000e-01,7.826000e-01,7.733000e-01,7.886000e-01,7.813000e-01
sf_tf02_player_rep_norm,4.664201e+06,4.650909e+06,4.650909e+06,4.650909e+06,4.686842e+06,4.655340e+06,4.650909e+06,4.650909e+06,4.650909e+06,4.650909e+06,4.656226e+06
sf_tf03_correlation_causation,2.769462e+07,2.769464e+07,2.310314e+07,2.769458e+07,2.769463e+07,2.310365e+07,2.196483e+07,4.779684e+06,2.311080e+07,2.769462e+07,9.365208e+06
sf_tf04_balance_torque_leakfree,1.000000e+00,1.000000e+00,1.000000e+00,NaN,NaN,1.000000e+00,NaN,NaN,1.000000e+00,NaN,NaN
sf_tf05_credit_card_leakage,9.659000e-01,9.144000e-01,9.659000e-01,9.568000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01,9.659000e-01
sf_tf06_metric_select_leakfree,9.798000e-01,9.796000e-01,9.803000e-01,NaN,NaN,9.223000e-01,NaN,NaN,9.809000e-01,NaN,NaN


In [9]:
summary = {"stage1_fix": s1.reset_index().to_dict("records"), "stage1_wrong": wr.reset_index().to_dict("records"),
  "fix_reject_instr": jr[(~jr.useful)&(jr.kind=="fix")].groupby("model").instr.agg(["sum","count"]).reset_index().to_dict("records"),
  "agreeing_judgements": jr[jr.kind=="wrong"].groupby(["model","useful"]).size().reset_index(name="n").to_dict("records"),
  "runs": {a: len(v) for a, v in recs.items()}, "stops": {a: dict(Counter(r["stop"] for r in v)) for a, v in recs.items()},
  "targeted": targeted.to_dict("records"), "targeted_pooled": pooled.round(3).to_dict(), "fisher_p": fisher, "cost_usd": cost,
  "score_mean": scores.to_dict("index")}
json.dump(summary, open(REPO / "quick_runs/results/P20/summary.json", "w"), indent=1, default=str)
jr.to_csv(REPO / "quick_runs/results/P20/judgements.csv", index=False)
"done" 

'done'